### Transform Sprints Data

1. Read bronze_sprints table
2. Keep only the columns required for analytics (Drop url column)
3. Standardise column names using snake_case (constructorId → constructor_id, driverId → driver_id, raceName → race_name, positionText → finish_position_text)
4. Rename columns to make them more meaningful (date → race_date, grid → grid_position, laps → completed_laps, number → car_number, position → finish_position)
5. Filter out rows where season, round, constructor_id, or driver_id is null (business key validation)
6. Remove duplicate records
7. Transform values of column race_name to Title Case
8. Write the transformed data to silver_sprints table

In [0]:
%run ../00-common/01.environment-configuration

In [0]:
bronze_table = F"{catalog_name}.{bronze_schema}.sprints"
silver_table = F"{catalog_name}.{silver_schema}.sprints"

In [0]:
sprints_df = spark.read.table(bronze_table)

In [0]:
from pyspark.sql import functions as F

In [0]:
sprints_selected_df = (
    sprints_df.select(
        F.col("date"),
        F.col("raceName"),
        F.col("round"),
        F.col("season"),
        F.col("constructorId"),
        F.col("driverId"),
        F.col("grid"),
        F.col("laps"),
        F.col("number"),
        F.col("points"),
        F.col("position"),
        F.col("positionText"),
        F.col("status"),
        F.col("timestamp"),
        F.col("source_file")
    )    
)

### Standardise column names using snake_case (constructorId → constructor_id, driverId → driver_id, raceName → race_name, positionText → finish_position_text)
### Rename columns to make them more meaningful (date → race_date, grid → grid_position, laps → completed_laps, number → car_number, position → finish_position)

In [0]:
sprints_renamed_df = (
    sprints_selected_df
        .withColumnsRenamed({
            "constructorId": "constructor_id",
            "driverId": "driver_id",
            "raceName": "race_name",
            "positionText": "finish_position_text",
            "date": "race_date",           
            "grid": "grid_position",
            "laps": "completed_laps",
            "number": "car_number",
            "position": "finish_position"
        })
)

In [0]:
display(sprints_renamed_df)

### Filter out rows where season, round, constructor_id, or driver_id is null (business key validation)

In [0]:
sprints_valid_df =(
    sprints_renamed_df
        .filter(
            F.col("season").isNotNull() &
            F.col("round").isNotNull() &
            F.col("constructor_id").isNotNull() &
            F.col("driver_id").isNotNull()
        )    
)        

In [0]:
display(sprints_valid_df)

In [0]:
display(sprints_renamed_df.count()-sprints_valid_df.count())

### Remove duplicate records

In [0]:
sprints_distinct_df = (
    sprints_valid_df
        .dropDuplicates(["season", "round", "driver_id", "constructor_id"])
)

In [0]:
display(sprints_valid_df.count()-sprints_distinct_df.count())

### Transform values of column race_name to Title Case

In [0]:
sprints_final_df = (
    sprints_distinct_df
        .withColumn("race_name", F.initcap(F.col("race_name")))
)

In [0]:
display(sprints_final_df)

In [0]:
(
    sprints_final_df
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(silver_table)
)

In [0]:
display(spark.table(silver_table))
